# 06. 어떤 기업에서 위축 신호가 더 강한가 — 채널 유형 · 자동이체 · 채널 구성 이동
> 계획서: `계획/P6_채널유형화_위축비교.md` (Codex 6차 반영) · 코드: `code/types_cnt.py`

**한 줄 요약**: P5에서 *"거래 건수가 직전 6개월보다 35% 넘게 줄어든 기업은 이후 6개월 입금이 크게 줄어들 가능성이 같은 조건의 기업보다 약 4.7배"* 였습니다. 이 노트북은 그 신호가 **누구에게서 더 뚜렷한지**를 쉬운 표와 그림으로 봅니다.

| 질문 | 보는 것 |
|---|---|
| **C1** 어떤 기업에서 신호가 잘 잡히나 | 채널 유형별 탐지 비율 (규모를 맞춰서) |
| **C2** 신호의 의미가 유형마다 같나 | 유형별 '입금 약화 가능성의 배수' (같은 유형 안에서 비교) |
| **C3** 자동이체가 많으면 덜 떠나나 | 자동이체 높음/낮음별 같은 배수 |
| **C4** 채널만 옮긴 기업은 위축과 다른가 | 총 활동은 그대로인데 오프라인 → 온라인으로 옮겨 간 기업의 입금 약화 |

**읽는 법**: 배수는 *"탐지된 기업이 입금이 크게 줄 가능성 ÷ 같은 조건의 비탐지 기업이 입금이 크게 줄 가능성"* 입니다. 괄호 안은 95% 범위(표본이 달라져도 이 안에 들어올 것으로 보는 구간)이며 **범위가 1을 포함하면 차이가 있다고 말하기 어렵습니다.** 모든 결과는 *연관*이고 원인이 아닙니다. 표본은 36개월 법인(P36)이 먼저이고, 35개월 법인(P35만)에서 같은 방향인지 마지막에 확인합니다.

**유형을 만드는 단계(§1)에서는 입금·탐지 결과를 결합하지 않습니다.** 화면 출력은 집계값뿐 · 5 미만 칸은 `<5` · 법인ID 없음. 몇 분 걸릴 수 있습니다 (`CNT_BOOT` 로 줄일 수 있음, 기본 2,000).

In [ ]:
import os, sys, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ── 설정: 여기만 고치세요 ────────────────────────────────────────────
WORK_DIR = os.environ.get("CNT_WORK_DIR") or None   # 기본값: 레포의 data/work_cnt (.gitignore 대상)
# ────────────────────────────────────────────────────────────────────

def _find_lib():
    for base in [Path.cwd(), *Path.cwd().parents]:
        for cand in (base / "code", base / "건수분석" / "code", base / "yangdaecheon" / "건수분석" / "code"):
            if (cand / "cntlib.py").exists():
                return cand
    raise SystemExit("cntlib.py 를 찾지 못했습니다. 노트북을 yangdaecheon/건수분석/notebooks 에서 여세요.")

sys.path.insert(0, str(_find_lib()))
import cntlib as cl
cl.setup_korean_font()
WORK = cl.resolve_work_dir(WORK_DIR)
OUT = WORK / "output"
print("산출물 폴더:", WORK)

In [ ]:
panel = cl.CountPanel.load(WORK / "interim" / "count_panel.npz")
_m = np.load(WORK / "interim" / "sample_masks.npz")
masks = {k: _m[k] for k in _m.files}
if panel.synthetic:
    print(cl.SYNTHETIC_NOTICE)
pops = {"P36": masks["P36"], "P35+": masks["P35p"]}
print({k: int(v.sum()) for k, v in pops.items()}, "개 기업")

import math
import detect_cnt as dc
import validate_cnt as vc
import types_cnt as tp

spec = json.load(open(WORK / "interim" / "detect_spec_v2.json", encoding="utf-8"))
rule = spec["rule"]
MODE, K, L, PHI2, N2 = rule["비교 기준"], int(rule["k"]), int(rule["L"]), float(rule["phi"]), float(rule["n_pct"])
B = int(os.environ.get("CNT_BOOT", "2000"))
print(f"고정 규칙(v2): 비교 기준 {MODE}, k={K}, Φ={PHI2:.2f}, n={N2:.0f}%  |  P5에서 정한 입금 약화 문턱·규모 경계·업종군을 그대로 씁니다")

def sup(x):
    return cl.sup(int(round(x)))

def f_rr(rr, lo, hi):
    return "-" if not np.isfinite(rr) else (f"{rr:.2f} ({lo:.2f}~{hi:.2f})" if np.isfinite(lo) else f"{rr:.2f}")

mk36, mk35 = masks["P36"], masks["P35"]
C5 = vc.fit_constants(panel, mk36, MODE, K, PHI2, N2)
D36 = vc.prepare(panel, mk36, C5)
assert int((D36["det"] >= 0).sum()) == spec["power"]["P36"]["N1"], "04의 탐지 기업 수와 다릅니다 (04를 다시 실행하세요)"
ER = D36["er"]
levels_sec = panel.attr_levels["업종_대분류"]
SECN = [levels_sec[c] for c in C5["top_codes"]] + ["기타"]
TN, AN = tp.TYPE_NAMES, tp.AUTO_NAMES
print(f"P36 판정 가능·탐지 준비 완료 (탐지 기업 {int((D36['det'] >= 0).sum()):,}곳)")

## 1. 유형 만들기 (입금·탐지 결과와 결합하지 않음)
- **프로파일 기간 = 2023년 상반기(1~6월)**: 가장 이른 판정 기준월(2024.06)의 비교 기준 구간이 2023.07부터라서, 유형을 만드는 건수와 탐지에 쓰는 건수가 **겹치지 않습니다.**
- **오프라인 비중** = (창구+ATM) ÷ (창구+인터넷+스마트+폰+ATM) 평균 근사 건수. 36개월 법인(P36)에서 **3등분 경계**를 한 번 정해(경계와 같으면 낮은 쪽) 35개월 법인에도 그대로 씁니다.
- **자동이체 수준** = 같은 기간 자동이체 평균 코드가 P36 중앙값 이상이면 높음.

In [ ]:
T36 = tp.make_types(panel, mk36)
typ, aut, nob = T36["type"], T36["auto_level"], T36["nobs"]
ok_t = typ >= 0
rows = [("프로파일 관측 달 ≥ 4 인 기업", cl.sup(int((nob >= tp.MIN_OBS).sum()))), ("유형 미정(관측 부족 또는 활동 0)", cl.sup(int((~ok_t).sum()))),
        ("오프라인 비중 경계 (33.3% / 66.7%)", f"{T36['type_edges'][0]:.3f} / {T36['type_edges'][1]:.3f}"),
        ("자동이체 평균 코드 중앙값(경계)", f"{T36['auto_edge']:.2f}")]
cl.show_table(pd.DataFrame(rows, columns=["항목", "값"]), "유형 정의의 기준값 (P36 에서 고정)", OUT / "nb06", "type_rule")
present = [t for t in range(3) if (typ == t).any()]
if len(present) < 3:
    print("※ 경계가 겹쳐 유형이 비었습니다 → 존재하는 유형만 비교합니다:", [TN[t] for t in present])

jf0 = (D36["status"][:, ER] == 0).any(axis=1)
size_f = tp.firm_detection_info(D36)[1]
ceil_first = dc.ceiling_at_first_judgeable(panel, mk36, D36["change"], PHI2)
fx_any = np.zeros(len(typ), dtype=bool)
for c in cl.FX:
    fx_any |= ((panel.cnt[c][mk36][:, 0:6] > 0) & panel.observed[mk36][:, 0:6]).any(axis=1)
rows = []
for t in present:
    m = typ == t
    mj = m & jf0
    r = {"유형": TN[t], "기업 수": cl.sup(m.sum()), "비중 %": round(100 * m.sum() / max(int(ok_t.sum()), 1), 1), "판정 가능 기업": cl.sup(mj.sum()),
         "판정 가능률 %": round(100 * mj.sum() / max(int(m.sum()), 1), 1), "천장 보유 % (판정 가능 기업)": round(100 * float(ceil_first[mj].mean()), 1) if mj.any() else None,
         "자동이체 높음 %": round(100 * float((aut[m] == 1).mean()), 1), "외환 이용 %": round(100 * float(fx_any[m].mean()), 1)}
    for q in range(5):
        r[f"규모{q + 1} %"] = round(100 * float((size_f[mj] == q).mean()), 1) if mj.any() else None
    rows.append(r)
cl.show_table(pd.DataFrame(rows), "유형별 크기와 구성 (판정 가능 기업, 입금·탐지 미결합)", OUT / "nb06", "type_context")
sec_f = D36["sector"]
rows = [{"유형": TN[t], **{nm: round(100 * float((sec_f[(typ == t) & jf0] == g).mean()), 1) for g, nm in enumerate(SECN)}} for t in present if ((typ == t) & jf0).any()]
cl.show_table(pd.DataFrame(rows), "유형별 업종군 구성 % (판정 가능 기업)", OUT / "nb06", "type_sector")
okA = aut >= 0
rows = [{"자동이체 수준": AN[a], "기업 수": cl.sup((aut == a).sum()), "판정 가능 기업": cl.sup(((aut == a) & jf0).sum()),
         **{f"규모{q + 1} %": round(100 * float((size_f[(aut == a) & jf0] == q).mean()), 1) for q in range(5)}} for a in (0, 1)]
cl.show_table(pd.DataFrame(rows), "자동이체 수준별 크기와 규모 구성", OUT / "nb06", "auto_context")
tie = float((T36["auto"][okA] == T36["auto_edge"]).mean())
print(f"자동이체 동률(중앙값과 같은 값을 가진 기업) 비율: {100 * tie:.1f}% → 같으면 '높음'에 넣었습니다")
rows = [{"다시 만드는 방식": f"cap {c:g}", "유형이 바뀐 기업 %": round(100 * tp.type_change_rate(panel, mk36, c), 1)} for c in (51.0, 100.0)]
cl.show_table(pd.DataFrame(rows), "'50건초과' 가정을 바꿔 유형을 처음부터 다시 만들면 몇 %가 다른 유형이 되나", OUT / "nb06", "type_cap")

## 2. 모든 비교를 한 번에 계산 (P36)
36개월 법인의 C1~C4를 한꺼번에 계산합니다 (몇 십 초). 같은 코드를 마지막에 35개월 법인에도 그대로 씁니다.

In [ ]:
t0 = time.time()
RES36 = tp.run_all(panel, mk36, D36, T36, C5, B=B, seed0=20261100)
print(f"P36 계산 완료 ({time.time() - t0:.0f}초)")

def rr_table(res, names, min_det=30):
    rows = []
    for gi, g in enumerate(res["levels"]):
        p, n1 = res["point"][gi], res["units"][gi]
        if p is None or n1 < min_det:
            rows.append({"집단": names[g], "탐지 기업": sup(n1), "배수(RR) (95% 범위)": "기술만 (탐지 30곳 미만 또는 비교군 없음)"})
            continue
        lo, hi, nv = tp.group_ci(res, gi)
        rows.append({"집단": names[g], "탐지 기업": sup(p["n1"]), "약화 사건(탐지군)": cl.sup(p["obs"]), "탐지군 약화율 %": round(100 * p["p_det"], 1),
                     "같은 집단 비탐지 기업 약화율 % (표준화)": round(100 * p["p_ctrl"], 1), "배수(RR) (95% 범위)": f_rr(p["rr"], lo, hi),
                     "공통 지지 제외율 %": round(100 * p["excluded_share"], 1)})
    return pd.DataFrame(rows)

def ratio_info(res, ga, gb):
    pa, pb = res["point"][ga], res["point"][gb]
    ra = pa["rr"] if pa is not None else math.nan
    rb = pb["rr"] if pb is not None else math.nan
    pt = tp.point_ratio(ra, rb)
    ci = tp.ratio_ci(res["boot"], ga, gb)
    return pt, ci["lo"], ci["hi"], ci["valid"]

## 3. C1 — 어떤 기업에서 위축 신호가 잘 잡히나 (규모를 맞춰서)
판정 가능한 기업 중 **한 번이라도 위축 신호(−35% 이하)가 잡힌 기업**의 비율입니다. 유형마다 기업 규모가 다르므로 **규모 5분위별 비율**을 먼저 보고, 전체 규모 분포로 맞춘 **'규모를 맞춘 탐지 비율'** 을 비교합니다.
**주의**: 이것은 *탐지 규칙이 얼마나 잘 잡느냐*이지 실제 위축이 얼마나 자주 생기느냐가 아닙니다 (거래가 아주 많은 기업은 건수 구간의 천장에 닿아 변화가 둔하게 보입니다).

In [ ]:
c1 = RES36["c1"]
sup1, pt1 = c1["support"], c1["point"]
n_tq, d_tq = sup1["n"], sup1["d"]
rows = []
for t in present:
    r = {"유형": TN[t]}
    for q in range(5):
        r[f"규모{q + 1}"] = (f"{cl.sup(d_tq[t, q])}/{cl.sup(n_tq[t, q])}" + (f" ({100 * d_tq[t, q] / n_tq[t, q]:.1f}%)" if n_tq[t, q] >= 10 else "")) if n_tq[t, q] > 0 else "-"
    rows.append(r)
cl.show_table(pd.DataFrame(rows), "유형 × 규모 5분위: 탐지된 기업 / 판정 가능 기업 (탐지율)", OUT / "nb06", "c1_type_size")
print("표준화에 쓴 규모 분위(모든 유형에서 기업 10곳 이상):", [q + 1 for q in sup1["Qs"]], "| 표준 인구 = 이 분위들의 전체 규모 분포")
rows = []
for t in present:
    v = c1["boot"][:, t]
    v = v[np.isfinite(v)]
    rows.append({"유형": TN[t], "규모를 맞춘 탐지 비율 %": round(100 * pt1[t], 1), "95% 범위 %": f"{100 * np.percentile(v, 2.5):.1f} ~ {100 * np.percentile(v, 97.5):.1f}" if v.size > 20 else "-",
                 "단순 탐지 비율 % (규모 안 맞춤)": round(100 * float(RES36["det_f"][RES36["jf"] & (typ == t)].mean()), 1)})
C1_TAB = pd.DataFrame(rows)
cl.show_table(C1_TAB, "C1 규모를 맞춘 탐지 비율", OUT / "nb06", "c1_standardized")
c1_ratio = None
if 0 in present and 2 in present:
    pt, lo, hi, nv = (tp.point_ratio(pt1[2], pt1[0]),) + tuple(tp.ratio_ci(c1["boot"], 2, 0).values())
    c1_ratio = (pt, lo, hi)
    print(f"창구·ATM 중심형 ÷ 인터넷·스마트 중심형 = {f_rr(pt, lo, hi)} (유효 반복 {nv})  →  " +
          ("범위가 1을 포함: 규모를 맞추면 탐지 비율 차이가 있다고 말하기 어렵습니다" if lo <= 1 <= hi else ("창구·ATM 중심형에서 신호가 더 잘 잡힘" if lo > 1 else "인터넷·스마트 중심형에서 신호가 더 잘 잡힘")))
fig, ax = plt.subplots(figsize=(6, 3))
xs = [TN[t] for t in present]
ys = [100 * pt1[t] for t in present]
errs = [[100 * (pt1[t] - np.nanpercentile(c1["boot"][:, t], 2.5)) for t in present], [100 * (np.nanpercentile(c1["boot"][:, t], 97.5) - pt1[t]) for t in present]]
ax.bar(xs, ys, yerr=errs, color="#2a78d6", alpha=0.85, capsize=4)
ax.set_ylabel("규모를 맞춘 탐지 비율 (%)"); ax.set_title("위축 신호가 잡히는 비율 (유형별, 막대 위 선 = 95% 범위)")
plt.tight_layout(); plt.show()

## 4. C2 — 신호의 의미가 유형마다 같은가
유형별로 **탐지된 기업**과 **같은 유형의 비탐지 기업**(같은 기준월·규모·업종)만 비교해 *"탐지됐을 때 입금이 크게 줄 가능성의 배수"* 를 구합니다 (전체 비탐지 기업과 비교하지 않는 이유: 유형 자체가 입금 약화 빈도를 바꿀 수 있어서). 유형 간 차이는 **두 배수의 비**(창구·ATM 중심형 ÷ 인터넷·스마트 중심형)로 봅니다.

In [ ]:
c2 = RES36["c2"]
C2_TAB = rr_table(c2, TN)
cl.show_table(C2_TAB, "C2 유형별 입금 약화 가능성의 배수 (탐지 기업 ÷ 같은 유형 비탐지 기업)", OUT / "nb06", "c2_type_rr")
C2_RATIOS = []
for ga, gb, lab in ((2, 0, "창구·ATM 중심형 ÷ 인터넷·스마트 중심형"), (1, 0, "혼합형 ÷ 인터넷·스마트 중심형"), (2, 1, "창구·ATM 중심형 ÷ 혼합형")):
    if ga in present and gb in present:
        pt, lo, hi, nv = ratio_info(c2, ga, gb)
        C2_RATIOS.append({"비교": lab, "배수의 비 (95% 범위)": f_rr(pt, lo, hi), "유효 반복": nv, "읽는 법": ("범위가 1을 포함: 차이가 있다고 말하기 어려움" if np.isfinite(lo) and lo <= 1 <= hi else
                                                                                   ("앞 유형에서 신호의 의미(입금 약화 배수)가 더 큼" if np.isfinite(lo) and lo > 1 else ("앞 유형에서 배수가 더 작음" if np.isfinite(hi) else "정의 불가")))})
cl.show_table(pd.DataFrame(C2_RATIOS), "C2 유형 간 차이: 두 배수의 비", OUT / "nb06", "c2_ratio")

## 5. C3 — 자동이체가 많은 기업은 덜 떠나나
같은 방식으로 **자동이체 높음/낮음**별 배수를 봅니다. '관계가 깊어 쉽게 떠나지 않는다'는 경험칙을 시험하지만, 자동이체가 많은 기업이 원래 안정적이거나 규모가 큰 것일 수도 있어 **방향을 정하지 않고 연관만** 봅니다.

In [ ]:
c3 = RES36["c3"]
C3_TAB = rr_table(c3, AN)
cl.show_table(C3_TAB, "C3 자동이체 수준별 입금 약화 가능성의 배수", OUT / "nb06", "c3_auto_rr")
pt3, lo3, hi3, nv3 = ratio_info(c3, 1, 0)
print(f"자동이체 높음 ÷ 낮음 = {f_rr(pt3, lo3, hi3)} (유효 반복 {nv3})  →  " +
      ("범위가 1을 포함: 차이가 있다고 말하기 어렵습니다" if np.isfinite(lo3) and lo3 <= 1 <= hi3 else
       ("자동이체가 많은 기업에서 신호의 의미(입금 약화 배수)가 더 큼" if np.isfinite(lo3) and lo3 > 1 else "자동이체가 많은 기업에서 배수가 더 작음 (덜 떠나는 쪽의 연관)" if np.isfinite(hi3) else "정의 불가")))

## 6. C4 — 채널만 옮긴 기업은 위축과 다른가
**총 활동은 유지한 채 오프라인(창구·ATM) 비중이 6개월 사이 0.2 이상 줄어든 기업**을 '채널 구성 이동'으로 봅니다. 위축 탐지(건수 −35% 이하)와는 **같은 달에 동시에 성립할 수 없는 별개의 신호**입니다. 비교는 *그 달 아직 이동도 위축 탐지도 되지 않은 기업*과 합니다. 총 활동 조건을 뺀 버전을 함께 봅니다. 이동 기업이 30곳 미만이거나 입금 약화 사건이 5건 미만이면 배수 없이 **기술만** 합니다.

In [ ]:
R_main = vc.analyze_units(D36["units"], D36["F"], B, seed=20261001)
rows = []
for key, lab in (("c4_kept", "채널 구성 이동 (총 활동 ±20% 유지)"), ("c4_any", "오프라인 비중 −0.2 이하 (총 활동 조건 없음, 위축 문턱 미만)")):
    c4 = RES36[key]
    r = {"집단": lab, "이동 기업": sup(c4["n_units"]), "약화 사건": cl.sup(c4["events"]), "비교 기업-월": cl.sup(c4["ctrl_units"]),
         "규모 분포(1~5분위)": " / ".join(cl.sup(x) for x in c4["size_dist"])}
    if c4["descriptive"]:
        r["배수(RR) (95% 범위)"] = "기술만 (이동 기업 30곳 미만 또는 약화 5건 미만)"
    else:
        R = c4["R"]
        r.update({"이동군 약화율 %": round(100 * R["p_det"], 1), "비교군 약화율 % (표준화)": round(100 * R["p_ctrl"], 1), "배수(RR) (95% 범위)": f_rr(R["rr"], R["lo"], R["hi"])})
    rows.append(r)
rows.append({"집단": "(참고) 위축 탐지군 — P5 결과", "이동 기업": sup(R_main["n1"]), "약화 사건": cl.sup(R_main["obs"]), "이동군 약화율 %": round(100 * R_main["p_det"], 1),
             "비교군 약화율 % (표준화)": round(100 * R_main["p_ctrl"], 1), "배수(RR) (95% 범위)": f_rr(R_main["rr"], R_main["lo"], R_main["hi"])})
C4_TAB = pd.DataFrame(rows)
cl.show_table(C4_TAB, "C4 채널 구성 이동 기업의 입금 약화 (위축 탐지군과 나란히)", OUT / "nb06", "c4_shift")

## 7. 한 장으로 보기 — 배수 비교 (P36)

In [ ]:
items = []
for gi, g in enumerate(c2["levels"]):
    p = c2["point"][gi]
    if p is not None and c2["units"][gi] >= 30:
        lo, hi, _ = tp.group_ci(c2, gi)
        items.append((TN[g], p["rr"], lo, hi))
for gi, g in enumerate(c3["levels"]):
    p = c3["point"][gi]
    if p is not None and c3["units"][gi] >= 30:
        lo, hi, _ = tp.group_ci(c3, gi)
        items.append((AN[g], p["rr"], lo, hi))
c4k = RES36["c4_kept"]
if not c4k["descriptive"]:
    items.append(("채널 구성 이동", c4k["R"]["rr"], c4k["R"]["lo"], c4k["R"]["hi"]))
items.append(("위축 탐지군 전체 (P5)", R_main["rr"], R_main["lo"], R_main["hi"]))
fig, ax = plt.subplots(figsize=(7, 0.55 * len(items) + 1.2))
for i, (lab, rr, lo, hi) in enumerate(items[::-1]):
    ax.plot([lo, hi], [i, i], color="#2a78d6", lw=3, solid_capstyle="round")
    ax.plot([rr], [i], "o", color="#eb6834", ms=8)
    ax.text(hi + 0.15, i, f"{rr:.1f}배", va="center", fontsize=9)
ax.axvline(1, color="k", ls="--", lw=1)
ax.set_yticks(range(len(items))); ax.set_yticklabels([x[0] for x in items[::-1]])
ax.set_xlabel("입금이 크게 줄 가능성의 배수 (점 = 추정, 선 = 95% 범위, 점선 = 1배)")
ax.set_title("위축 신호가 잡혔을 때: 집단별 입금 약화 배수 (P36)")
plt.tight_layout(); plt.show()

## 8. P35만 확인 — 같은 정의·같은 경계로
35개월 법인(P36과 겹치지 않는 기업)에서 **P36에서 정한 경계값 그대로** 같은 비교를 합니다. P36에서 눈에 띄었는지와 상관없이 **모든 비교를 나란히** 보여 줍니다. P35만은 한 달이 빠진 기업이라 구성이 다를 수 있어 구성 표를 함께 봅니다.

In [ ]:
t0 = time.time()
D35 = vc.prepare(panel, mk35, C5)
T35 = tp.make_types(panel, mk35, T36["type_edges"], T36["auto_edge"])
RES35 = tp.run_all(panel, mk35, D35, T35, C5, B=B, seed0=20261200)
print(f"P35만 계산 완료 ({time.time() - t0:.0f}초)")
typ35 = T35["type"]
jf35 = (D35["status"][:, D35["er"]] == 0).any(axis=1)
sz35 = tp.firm_detection_info(D35)[1]
rows = [{"표본": nm, "유형 정의 기업": cl.sup((tv >= 0).sum()), **{TN[t]: round(100 * float((tv[tv >= 0] == t).mean()), 1) for t in range(3)},
         **{f"규모{q + 1} % (판정 가능)": round(100 * float((szv[jv] == q).mean()), 1) for q in range(5)}}
        for nm, tv, jv, szv in (("P36", typ, jf0, size_f), ("P35만", typ35, jf35, sz35))]
cl.show_table(pd.DataFrame(rows), "표본 구성 비교: 유형 비중(%)과 규모 분포(%)", OUT / "nb06", "p35_context")

In [ ]:
def verdict(pt36, lo36, hi36, pt35):
    if not np.isfinite(pt36) or not np.isfinite(lo36):
        return "정의 불가(기술만)"
    if lo36 <= 1 <= hi36:
        return "뚜렷한 차이 없음 (P36 범위가 1을 포함)"
    if not np.isfinite(pt35):
        return "P35만에서 정의 불가 → 확인 안 됨"
    return "일관된 차이 (P36 범위가 1을 벗어나고 P35만도 같은 방향)" if (pt36 > 1) == (pt35 > 1) else "불확실 (P35만에서 방향이 다름)"

rows = []
def add(label, p36, p35):
    rows.append({"비교": label, "P36 (95% 범위)": f_rr(*p36[:3]), "P35만 (95% 범위)": f_rr(*p35[:3]), "판정": verdict(p36[0], p36[1], p36[2], p35[0])})

def ratio_from(res, ga, gb):
    try:
        pt, lo, hi, nv = ratio_info(res, ga, gb)
    except Exception:
        return (math.nan, math.nan, math.nan)
    return (pt, lo, hi)

if 0 in present and 2 in present:
    t35p = RES35["c1"]["point"]
    ci35 = tp.ratio_ci(RES35["c1"]["boot"], 2, 0)
    add("C1 규모를 맞춘 탐지 비율: 창구·ATM 중심형 ÷ 인터넷·스마트 중심형", (tp.point_ratio(pt1[2], pt1[0]), *tuple(tp.ratio_ci(c1["boot"], 2, 0).values())[:2]),
        (tp.point_ratio(t35p[2], t35p[0]), ci35["lo"], ci35["hi"]))
    add("C2 유형별 배수의 비: 창구·ATM 중심형 ÷ 인터넷·스마트 중심형", ratio_from(c2, 2, 0), ratio_from(RES35["c2"], 2, 0))
    add("C2 유형별 배수의 비: 혼합형 ÷ 인터넷·스마트 중심형", ratio_from(c2, 1, 0), ratio_from(RES35["c2"], 1, 0))
add("C3 자동이체 높음 ÷ 낮음 (배수의 비)", ratio_from(c3, 1, 0), ratio_from(RES35["c3"], 1, 0))
def shift_triplet(res):
    c = res["c4_kept"]
    return (c["R"]["rr"], c["R"]["lo"], c["R"]["hi"]) if not c["descriptive"] else (math.nan, math.nan, math.nan)
add("C4 채널 구성 이동 기업의 입금 약화 배수 (1 = 차이 없음)", shift_triplet(RES36), shift_triplet(RES35))
OVERALL = pd.DataFrame(rows)
cl.show_table(OVERALL, "비교 3개(+C1) 한눈에: P36 과 P35만, 같은 정의", OUT / "nb06", "overall")
rows = []
for nm, res in (("P36", RES36), ("P35만", RES35)):
    for gi, g in enumerate(res["c2"]["levels"]):
        p = res["c2"]["point"][gi]
        if p is not None:
            lo, hi, _ = tp.group_ci(res["c2"], gi)
            rows.append({"표본": nm, "집단": TN[g], "탐지 기업": sup(p["n1"]), "배수(RR) (95% 범위)": f_rr(p["rr"], lo, hi)})
    for gi, g in enumerate(res["c3"]["levels"]):
        p = res["c3"]["point"][gi]
        if p is not None:
            lo, hi, _ = tp.group_ci(res["c3"], gi)
            rows.append({"표본": nm, "집단": AN[g], "탐지 기업": sup(p["n1"]), "배수(RR) (95% 범위)": f_rr(p["rr"], lo, hi)})
    c = res["c4_kept"]
    rows.append({"표본": nm, "집단": "채널 구성 이동", "탐지 기업": sup(c["n_units"]), "배수(RR) (95% 범위)": f_rr(*shift_triplet(res)) if not c["descriptive"] else "기술만"})
cl.show_table(pd.DataFrame(rows), "집단별 배수: P36 과 P35만", OUT / "nb06", "groups_both")
print("※ '뚜렷한 차이 없음'은 차이가 없다는 뜻이 아니라 이 자료로는 말하기 어렵다는 뜻입니다.")

## 9. 요약 저장

In [ ]:
def pk(p):
    return None if p is None else {k: (None if not np.isfinite(p[k]) else float(p[k])) for k in ("rr", "p_det", "p_ctrl", "n1", "obs", "excluded_share")}

def grp(res, names):
    out = {}
    for gi, g in enumerate(res["levels"]):
        d = pk(res["point"][gi])
        if d is not None:
            lo, hi, nv = tp.group_ci(res, gi)
            d.update({"lo": None if not np.isfinite(lo) else float(lo), "hi": None if not np.isfinite(hi) else float(hi), "valid": int(nv)})
        out[names[g]] = d
    return out

summary = {"type_edges": [float(x) for x in T36["type_edges"]], "auto_edge": float(T36["auto_edge"]), "present_types": [TN[t] for t in present],
           "P36": {"c2": grp(RES36["c2"], TN), "c3": grp(RES36["c3"], AN),
                   "c1": {TN[t]: float(pt1[t]) for t in present if np.isfinite(pt1[t])},
                   "c4_kept": {"n_units": RES36["c4_kept"]["n_units"], "events": RES36["c4_kept"]["events"], "descriptive": bool(RES36["c4_kept"]["descriptive"])}},
           "P35": {"c2": grp(RES35["c2"], TN), "c3": grp(RES35["c3"], AN),
                   "c1": {TN[t]: float(RES35["c1"]["point"][t]) for t in range(3) if np.isfinite(RES35["c1"]["point"][t])},
                   "c4_kept": {"n_units": RES35["c4_kept"]["n_units"], "events": RES35["c4_kept"]["events"], "descriptive": bool(RES35["c4_kept"]["descriptive"])}},
           "synthetic": bool(panel.synthetic)}
cl.save_json(WORK / "interim" / "nb06_summary.json", summary)
print("저장: interim/nb06_summary.json,", "output/nb06/*.csv")

## 마치면
실행 후 **저장(Ctrl+S)** 만 해 주세요. 결과 표는 `data/work_cnt/output/nb06/`에 자동 저장되어 Claude가 직접 읽고 해석합니다.